# 自动求导

In [1]:
import torch
x = torch.arange(4.0,requires_grad=True)
# requires_grad=True表示需要计算梯度
# x = torch.arange(4.0)
# x.requires_grad_(True)
x

tensor([0., 1., 2., 3.], requires_grad=True)

In [2]:
x.grad # 默认值是None

In [3]:
y = 2 * torch.dot(x,x)
y # 标量函数

tensor(28., grad_fn=<MulBackward0>)

In [4]:
y.backward()
x.grad, x.grad == 4 * x # ∂y/∂x = 4x  y关于x的导数存放在x.grad里

(tensor([ 0.,  4.,  8., 12.]), tensor([True, True, True, True]))

In [5]:
x.grad.zero_() # 梯度清零
y = x.sum() # y = x_1 + x_2 + x_3 + x_4
y.backward()
x.grad, x.grad == 1

(tensor([1., 1., 1., 1.]), tensor([True, True, True, True]))

In [6]:
x.grad.zero_()
y = x * x 
y.sum().backward() 
# .backward 只能反向求标量，所以对y求和
# 等价于y.backward(torch.ones(len(x)))
# y = Σ(x_i)^2
x.grad, x.grad == 2 * x

(tensor([0., 2., 4., 6.]), tensor([True, True, True, True]))

In [7]:
x.grad.zero_()
y = x * x # y_i = (x_i)^2
u = y.detach() 
# 从计算图中分离，创建一个不需要梯度的常量u
z = u * x # z_i = u_i * x_i

z.sum().backward()
# u被分离，视为常数
x.grad, x.grad == u # ∂z/∂x = u

(tensor([0., 1., 4., 9.]), tensor([True, True, True, True]))

In [8]:
x.grad.zero_()
y.sum().backward() # y_i = Σ(x_i)^2
x.grad == 2 * x

tensor([True, True, True, True])

In [9]:
def func(a):
    '''
    c 和 b 保持线性关系
    c = func(a) = k * a
    '''
    b = a * 2
    while b.norm() < 1000:
        b = b * 2
    if b.sum() > 0:
        c = b
    else:
        c = 100 * b
    return c

a = torch.randn(size=(), requires_grad=True)
d = func(a)
d.backward()

a.grad == d / a

tensor(True)

$$ L=10-\left(w_{3} w_{1} x+w_{4} w_{2} x\right)^{2} $$

$$ a=w_{1} x ; \quad b=w_{2} x ; \quad c=w_{3} a+w_{4} b ; \quad d=c^{2} $$

$$
\begin{aligned}
\frac{\partial L}{\partial x} &=\frac{\partial L}{\partial d} \frac{\partial d}{\partial c}\left(\frac{\partial c}{\partial a} \frac{\partial a}{\partial x}+\frac{\partial c}{\partial b} \frac{\partial b}{\partial x}\right) \\
 &=\frac{\partial L}{\partial d} \frac{\partial d}{\partial c} \frac{\partial c}{\partial a} \frac{\partial a}{\partial x}+\frac{\partial L}{\partial d} \frac{\partial d}{\partial c} \frac{\partial c}{\partial b} \frac{\partial b}{\partial x} \\
 &=(-1) \cdot 2 \cdot w_{3} \cdot w_{1}+(-1) \cdot 2 \cdot w_{4} \cdot w_{2}
\end{aligned}
$$

In [10]:
x = torch.randn((3,3),requires_grad=True)

w1 = torch.randn((3,3),requires_grad=True)
w2 = torch.randn((3,3),requires_grad=True)
w3 = torch.randn((3,3),requires_grad=True)
w4 = torch.randn((3,3),requires_grad=True)

a = w1 * x # 3 by 3
b = w2 * x # 3 by 3
c = w3 * a + w4 * b # 3 by 3
d = c * c # 3 by 3

a.retain_grad()
b.retain_grad()
c.retain_grad()
d.retain_grad()
L = (10 - d).sum()
L

tensor(53.0011, grad_fn=<SumBackward0>)

In [11]:
if x.grad is not None:
    x.grad.zero_()
if w1.grad is not None:
    w1.grad.zero_()
if w2.grad is not None:
    w2.grad.zero_()
if w3.grad is not None:
    w3.grad.zero_()
if w4.grad is not None:
    w4.grad.zero_()

L.backward(retain_graph=True)
a.grad, a.is_leaf

(tensor([[-3.5667e-02,  5.5984e-01,  1.7581e+01],
         [ 1.7303e-02, -1.5881e-01,  4.0524e-01],
         [ 1.4169e-01,  2.7046e-02, -2.4637e+00]]),
 False)

In [12]:
b.grad, b.is_leaf

(tensor([[-3.9744e-02, -5.2455e-01, -2.3534e+01],
         [-2.4957e-02,  1.9776e-02, -8.1886e-01],
         [ 2.2135e+00,  3.4537e-02, -1.1951e+00]]),
 False)

In [13]:
c.grad, c.is_leaf

(tensor([[ -0.0916,  -0.7975, -11.6812],
         [ -0.1868,  -0.0991,   2.2968],
         [ -1.6092,   0.0365,   1.7295]]),
 False)

In [14]:
d.grad, d.is_leaf

(tensor([[-1., -1., -1.],
         [-1., -1., -1.],
         [-1., -1., -1.]]),
 False)

In [15]:
x.grad, x.is_leaf, w1.is_leaf,  w2.is_leaf,  w3.is_leaf, w4.is_leaf

(tensor([[-9.5661e-03,  7.6933e-01,  4.6268e+01],
         [ 1.6257e-02, -8.0594e-03,  2.3854e+00],
         [-2.3221e+00,  7.8485e-04, -2.4599e+00]]),
 True,
 True,
 True,
 True,
 True)

# 线性回归

In [16]:
import torch
import numpy as np

In [17]:
def generate_data(true_w, true_b, num_samples=1000):
    """
    生成二维特征和一维标签的人工数据集
    真实模型: y = Xw + b + 噪声
    """
    X = torch.randn(num_samples, 2) # 生成特征数据 (1000个样本, 2个特征)  
    y = torch.matmul(X, true_w) + true_b # 生成标签
    y += torch.randn(y.shape) * 0.01 # 添加1%高斯噪声
    
    return X, y

In [18]:
def data_iter(X, y, batch_size=10):
    """
    生成小批量数据的迭代器
    参数:
        X: 特征张量 (num_samples, num_features)
        y: 标签张量 (num_samples,)
        batch_size: 每批数据量
    """
    num_samples = X.shape[0]
    indices = list(range(num_samples))
    
    np.random.shuffle(indices) # 随机打乱索引
    
    # 分批生成数据
    for i in range(0, num_samples, batch_size): # 0到num_samples每次步进batch_size个样本
        batch_indices = indices[i:min(i + batch_size, num_samples)] # 计算当前批次的索引范围
        yield X[batch_indices], y[batch_indices] # 逐步产生数据而不占用大量内存

In [19]:
def linreg(X, w, b):
    """线性回归模型: y = X*w + b"""
    return torch.matmul(X, w) + b

In [20]:
def squared_loss(y_hat, y):
    """
    均方误差损失函数
    参数:
        y_hat: 预测值
        y: 真实标签
    """
    return (y_hat - y.reshape(y_hat.shape)) ** 2 / 2

In [21]:
def sgd(params, lr):
    """
    小批量随机梯度下降
    参数:
        params: 包含需要优化的参数的列表 [w, b]
        lr: 学习率
    """
    with torch.no_grad():  # 禁用梯度计算
        for param in params:
            param -= lr * param.grad  # 参数更新
            param.grad.zero_()  # 梯度清零

In [22]:
 # 生成数据集
true_w = torch.tensor([2.0, -3.4]) # 设置真实权重和偏置
true_b = 4.2
X, y = generate_data(true_w, true_b, num_samples=1000)
    
 # 初始化模型参数
w = torch.normal(0, 0.01, size=(2,), requires_grad=True)
b = torch.zeros(1, requires_grad=True)
    
# 超参数设置
lr = 0.03  # 学习率
num_epochs = 10  # 训练轮数
batch_size = 32  # 批量大小
    
# 训练循环
for epoch in range(num_epochs):
    for X_batch, y_batch in data_iter(X, y, batch_size):
        y_hat = linreg(X_batch, w, b) # 前向传播
        loss = squared_loss(y_hat, y_batch) # 计算损失
        loss.sum().backward()
        sgd([w, b], lr) # 参数更新
    print(f'Epoch {epoch+1}, Loss {loss.mean():.6f}')
        
    
print(f'真实参数: w=[2.0, -3.4], b=4.2')
print(f'训练结果: w=[{w[0].item():.4f}, {w[1].item():.4f}], b={b.item():.4f}')

Epoch 1, Loss 0.000087
Epoch 2, Loss 0.000050
Epoch 3, Loss 0.000120
Epoch 4, Loss 0.000064
Epoch 5, Loss 0.000062
Epoch 6, Loss 0.000057
Epoch 7, Loss 0.000067
Epoch 8, Loss 0.000024
Epoch 9, Loss 0.000055
Epoch 10, Loss 0.000072
真实参数: w=[2.0, -3.4], b=4.2
训练结果: w=[1.9986, -3.4009], b=4.2009


# 线性回归(单层神经网络)

In [23]:
import torch
from torch.utils import data
import torch.nn as nn # neutral network
import numpy as np

In [24]:
def load_array(data_arrays, batch_size, is_train=True):
    """构造一个PyTorch数据迭代器"""
    # 将特征和标签组合成数据集
    dataset = data.TensorDataset(*data_arrays)
     # 创建数据加载器，shuffle表示是否打乱数据
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

In [25]:
true_w = torch.tensor([2.0, -3.4]) # 设置真实权重和偏置
true_b = 4.2
features, labels = generate_data(true_w, true_b, num_samples=1000) # 生成数据
labels = labels.reshape(-1, 1) # 列向量 100 by 1
# 模型输出形状[batch_size, 1]的二维张量，MSELoss要求输入与目标形状一致

batch_size = 32
data_iter = load_array((features, labels), batch_size) # 创建数据迭代器

next(iter(data_iter)) # 检查第一个批次的数据，第一个张量是特征，第二个张量是标签

[tensor([[ 0.3125,  1.0233],
         [-0.4089, -0.3066],
         [-0.6651, -1.3488],
         [ 0.3574, -1.9739],
         [ 0.0159,  1.4723],
         [ 2.2117,  0.1123],
         [-0.2602, -0.6055],
         [ 0.1123, -0.5458],
         [-0.4468, -0.9068],
         [-2.4239,  0.1771],
         [ 1.2370,  0.1625],
         [ 0.3255, -0.9345],
         [-0.5455,  0.3718],
         [ 0.9283, -0.6713],
         [ 1.6393,  0.9661],
         [-0.3880, -0.1984],
         [ 0.7451, -1.4341],
         [-1.8187,  0.0154],
         [ 1.8222,  0.0412],
         [ 0.4813,  1.0963],
         [ 0.3464,  0.4105],
         [ 0.4358, -0.0475],
         [ 0.7209, -1.1955],
         [ 1.0713,  1.2777],
         [-2.8239,  0.1923],
         [ 0.8188,  0.1828],
         [-0.2303,  1.7974],
         [ 1.9829, -2.0174],
         [ 0.3899,  0.4283],
         [-1.3595,  0.4825],
         [ 1.2187, -1.0733],
         [-0.6266,  1.4808]]),
 tensor([[ 1.3547],
         [ 4.4168],
         [ 7.4487],
         [

In [26]:
# 创建神经网络模型：2个输入特征，1个输出
net = nn.Sequential(nn.Linear(2,1))
# 模型输入形状[batch_size, 2]的二维张量，输出形状[batch_size, 1]的二维张量

# 初始化模型参数
# 权重初始化为均值为0，标准差为0.01的正态分布
net[0].weight.data.normal_(0,0.01)
# 偏置初始化为0
net[0].bias.data.fill_(0)

tensor([0.])

In [27]:
loss = nn.MSELoss() # 均方误差损失函数
trainer = torch.optim.SGD(net.parameters(), lr=0.03) # 随机梯度下降优化器，学习率为0.03

In [28]:
# 训练循环
num_epochs = 10  # 训练轮数
for epoch in range(num_epochs):
    for X_batch, y_batch in data_iter:
        y_hat = net(X_batch) # 前向传播计算预测值
        ls = loss(y_hat, y_batch) # 计算损失
        trainer.zero_grad() #清空梯度
        ls.backward() #反向传播计算梯度
        trainer.step() # 更新参数
    ls = loss(net(features), labels)
    print(f'Epoch {epoch+1}, Loss {ls.item():.6f}') # 单元素的张量使用.item()转换为标量
    
print(f'真实参数: w=[2.0, -3.4], b=4.2')
print(f'训练结果: w=[{w[0].item():.4f}, {w[1].item():.4f}], b={b.item():.4f}')

Epoch 1, Loss 0.670709
Epoch 2, Loss 0.013228
Epoch 3, Loss 0.000378
Epoch 4, Loss 0.000107
Epoch 5, Loss 0.000102
Epoch 6, Loss 0.000102
Epoch 7, Loss 0.000102
Epoch 8, Loss 0.000102
Epoch 9, Loss 0.000102
Epoch 10, Loss 0.000102
真实参数: w=[2.0, -3.4], b=4.2
训练结果: w=[1.9986, -3.4009], b=4.2009
